# Wine Quality Prediction - Updated Balanced Model

This notebook improves the original approach by combining a balanced quality-band classifier with a weighted Random Forest regressor so low-quality wines are less likely to be pulled toward the average.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.utils import resample

In [ ]:
DATA_PATH = Path('WineQT - WineQT.csv.csv')
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('Shape before duplicate removal:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Shape after duplicate removal:', df.shape)

In [ ]:
df['quality_label'] = df['quality'].apply(lambda value: 'Low' if value <= 4 else ('Medium' if value <= 6 else 'High'))
print(df['quality'].value_counts().sort_index())
print(df['quality_label'].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
df['quality'].value_counts().sort_index().plot(kind='bar', color='#c58b37')
plt.title('Wine Quality Distribution')
plt.xlabel('Quality')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.drop(columns=['quality_label']).corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
feature_columns = [column for column in df.columns if column not in ['quality', 'quality_label']]
X = df[feature_columns]
y_score = df['quality']
y_label = df['quality_label']

X_train, X_test, y_train_score, y_test_score, y_train_label, y_test_label = train_test_split(
    X,
    y_score,
    y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
train_df = X_train.copy()
train_df['quality'] = y_train_score.values
train_df['quality_label'] = y_train_label.values

max_class_size = train_df['quality_label'].value_counts().max()
balanced_parts = []
for label, group in train_df.groupby('quality_label'):
    balanced_parts.append(resample(group, replace=True, n_samples=max_class_size, random_state=42))

balanced_train_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)
print(balanced_train_df['quality_label'].value_counts())

In [ ]:
scaler = StandardScaler()
X_train_classifier = balanced_train_df[feature_columns]
y_train_classifier = balanced_train_df['quality_label']
X_train_classifier_scaled = scaler.fit_transform(X_train_classifier)
X_train_regressor_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced_subsample',
    random_state=42
)
classifier.fit(X_train_classifier_scaled, y_train_classifier)

quality_counts = y_train_score.value_counts().to_dict()
regression_weights = y_train_score.map(lambda value: 1.0 / quality_counts[value]).to_numpy()
regression_weights = regression_weights / regression_weights.mean()

regressor = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42
)
regressor.fit(X_train_regressor_scaled, y_train_score, sample_weight=regression_weights)

In [ ]:
predicted_labels = classifier.predict(X_test_scaled)
raw_score_predictions = regressor.predict(X_test_scaled)

def calibrate_score(score, label):
    if label == 'Low':
        return min(score, 4.4)
    if label == 'Medium':
        return min(max(score, 4.6), 6.4)
    return max(score, 6.6)

final_score_predictions = np.array([calibrate_score(score, label) for score, label in zip(raw_score_predictions, predicted_labels)])
final_score_predictions[:10]

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_score, final_score_predictions))
r2 = r2_score(y_test_score, final_score_predictions)
weighted_f1 = f1_score(y_test_label, predicted_labels, average='weighted')

print(f'RMSE: {rmse:.4f}')
print(f'R2 Score: {r2:.4f}')
print(f'Weighted F1 Score: {weighted_f1:.4f}')
print(f'Min predicted score: {final_score_predictions.min():.3f}')
print(f'Max predicted score: {final_score_predictions.max():.3f}')
print(f'Predictions below 5: {(final_score_predictions < 5).sum()}')

In [ ]:
print(classification_report(y_test_label, predicted_labels))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_score, final_score_predictions, alpha=0.6)
plt.xlabel('Actual Quality')
plt.ylabel('Predicted Quality')
plt.title('Actual vs Predicted Quality')
plt.show()

In [ ]:
prediction_error = y_test_score - final_score_predictions

plt.figure(figsize=(7, 5))
plt.hist(prediction_error, bins=40, color='#c58b37', edgecolor='black')
plt.title('Prediction Error Distribution')
plt.xlabel('Actual - Predicted')
plt.ylabel('Frequency')
plt.show()

In [ ]:
label_order = ['Low', 'Medium', 'High']
cm = confusion_matrix(y_test_label, predicted_labels, labels=label_order)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order).plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Quality Band Confusion Matrix')
plt.show()

In [ ]:
feature_importance = pd.Series(regressor.feature_importances_, index=feature_columns).sort_values()

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='barh', color='#c58b37')
plt.title('Feature Importance (Regressor)')
plt.xlabel('Importance')
plt.show()

In [ ]:
model_bundle = {
    'scaler': scaler,
    'classifier': classifier,
    'regressor': regressor,
    'feature_columns': feature_columns,
    'label_order': ['Low', 'Medium', 'High'],
    'calibration_rules': {
        'Low': {'max': 4.4},
        'Medium': {'min': 4.6, 'max': 6.4},
        'High': {'min': 6.6}
    }
}
joblib.dump(model_bundle, 'wine_quality_balanced_model.joblib')
print('Saved updated model to wine_quality_balanced_model.joblib')